In [2]:
! pip install mysql-connector-python

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   --- ------------------------------------ 1.6/16.5 MB 13.9 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/16.5 MB 13.1 MB/s eta 0:00:01
   --------- ------------------------------ 3.9/16.5 MB 13.1 MB/s eta 0:00:01
   ---------- ----------------------------- 4.2/16.5 MB 5.9 MB/s eta 0:00:03
   ---------- ----------------------------- 4.5/16.5 MB 5.2 MB/s eta 0:00:03
   -------------- ------------------------- 6.0/16.5 MB 5.1 MB/s eta 0:00:03
   --------------- ------------------------ 6.3/16.5 MB 4.9 MB/s eta 0:00:03
   --------------- ------------------------ 6.6/16.5 MB 4.4 MB/s eta 0:00:03
   --------------- ------------------------ 6.6/16.5 MB 4.4 MB/s eta 0:00:03
   --------------- ------------------------ 6.6/16.5 MB 4.4 MB/s eta 0:00:03
   --------------- ------------------------ 6.6/16.5 MB 4.4 MB/s eta 0:00:03
   --------------- ------------------------ 6.6/16.5 MB 4.4 MB/s eta 0:00:03
   


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import pandas as pd
import numpy as np
import mysql.connector

In [39]:
conn = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "root",
    database = "new_schema"
)
print("Connection successful")

Connection successful


In [40]:
conn.is_connected()

True

In [41]:
# creating a function out of it
def connect_to_db():
    return  mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "root",
    database = "new_schema"
)

In [42]:
connect_to_db().is_connected()

True

In [43]:
db = connect_to_db()
cursor = db.cursor(dictionary=True)

In [44]:
cursor

In [45]:
cursor.execute("select count(*) as Total_Supplier from suppliers")

In [46]:
row = cursor.fetchone()

In [47]:
list(row.values())[0]

50

In [48]:
def get_bacic_info(cursor):


    queries = {
    "Total Suppliers" : "select count(*) as Total_Supplier from suppliers",

    "Total products" : "select count(*) as total_products from products",

    "Total Category Dealing" : "select count(distinct category) as total_categories from products",

    "Total sales made in last 3 months (quantity * price)" : """ select 
    round(sum(abs(se.change_quantity) * p.price),2) as Total_sales_value_last_3_months
    from stock_entries se
    join products p 
    on p.product_id = se.product_id
    where se.change_type = "Sale"
    and
    se.entry_date >= (select date_sub(max(entry_date), interval 3 month) from stock_entries) 
    """,



    "Total restock made in last 3 months (quantity * price)" : """ 
    select 
    round(sum(abs(se.change_quantity) * p.price),2) as Total_sales_value_last_3_months
    from stock_entries se
    join products p 
    on p.product_id = se.product_id
    where se.change_type = "Restock"
    and
    se.entry_date >= (select date_sub(max(entry_date), interval 3 month) from stock_entries)
    """,


    "Below Reorder and No Pending Reorders" : """ 
    select count(*) 
    from products p 
    where p.stock_quantity < p.reorder_level
    and product_id not in 
    (
    select distinct product_id from reorders where status = "Pending"
    )"""

    }
    
    result = {}
    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        result[label] = list(row.values())[0]

    return result

In [49]:
def get_additional_tables(cursor):

    queries  = {
    "Suppliers and their contact details" : "select supplier_name, contact_name, email, phone from suppliers", 

    "Product with their suppliers and current stock" : """ select 
    p.product_name,
    s.supplier_name,
    p.stock_quantity,
    p.reorder_level
    from products p 
    join suppliers s
    on s.supplier_id = p.supplier_id
    order by p.product_name ASC
    """,

    "Product needing Reorder" : """select 
    product_id,
    product_name,
    stock_quantity,
    reorder_level
    from products 
    where stock_quantity < reorder_level
    """
    }

    table = {}
    for label, query in queries.items():
        cursor.execute(query)
        table[label] = cursor.fetchall()
    return table

In [50]:
def add_new_manual_id(cursor, db, p_name, p_category, p_price, p_stock, p_reorder, p_supplier):
    procedure_call = "call AddNewproductManualID(%s, %s,%s,%s,%s,%s)"
    params = (p_name, p_category, p_price, p_stock, p_reorder, p_supplier)
    cursor.execute(procedure_call, params)
    db.commit()

In [51]:
def get_categories(cursor):
    cursor.execute("select Distinct category from products order by category asc")
    rows = cursor.fetchall()
    return [row["category"] for row in rows]

In [52]:
get_categories(cursor)

['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']

In [53]:
def get_suppliers(cursor):
    cursor.execute("select supplier_id, supplier_name from suppliers order by supplier_name asc")
    return cursor.fetchall()

In [54]:
get_suppliers(cursor)

[{'supplier_id': 1, 'supplier_name': 'Anderson-Thompson'},
 {'supplier_id': 48, 'supplier_name': 'Armstrong-Vance'},
 {'supplier_id': 24, 'supplier_name': 'Barker, White and Carson'},
 {'supplier_id': 25, 'supplier_name': 'Barrett Ltd'},
 {'supplier_id': 3, 'supplier_name': 'Baxter-Meadows'},
 {'supplier_id': 19, 'supplier_name': 'Charles Inc'},
 {'supplier_id': 41, 'supplier_name': 'Clark Group'},
 {'supplier_id': 23, 'supplier_name': 'Douglas Ltd'},
 {'supplier_id': 32, 'supplier_name': 'Elliott-Ayers'},
 {'supplier_id': 7, 'supplier_name': 'Evans Inc'},
 {'supplier_id': 27, 'supplier_name': 'Franklin, Kane and Price'},
 {'supplier_id': 15, 'supplier_name': 'Freeman-Gordon'},
 {'supplier_id': 13, 'supplier_name': 'Gallagher-Miller'},
 {'supplier_id': 17, 'supplier_name': 'Gomez PLC'},
 {'supplier_id': 40, 'supplier_name': 'Hall-Brown'},
 {'supplier_id': 38, 'supplier_name': 'Harris-Cummings'},
 {'supplier_id': 42, 'supplier_name': 'Henderson LLC'},
 {'supplier_id': 50, 'supplier_name

In [55]:
def get_all_products(cursor):
    cursor.execute("select product_id, product_name from products order by product_name")
    return cursor.fetchall()

In [56]:
get_all_products(cursor)

[{'product_id': 13, 'product_name': 'Ability Snack'},
 {'product_id': 64, 'product_name': 'Account Toy'},
 {'product_id': 69, 'product_name': 'Actually Toy'},
 {'product_id': 48, 'product_name': 'After Table'},
 {'product_id': 112, 'product_name': 'Against Table'},
 {'product_id': 81, 'product_name': 'Alone Toy'},
 {'product_id': 87, 'product_name': 'Already Snack'},
 {'product_id': 40, 'product_name': 'Among Toy'},
 {'product_id': 11, 'product_name': 'Amount Snack'},
 {'product_id': 2, 'product_name': 'And Table'},
 {'product_id': 105, 'product_name': 'Another Device'},
 {'product_id': 129, 'product_name': 'Appear Toy'},
 {'product_id': 179, 'product_name': 'As Toy'},
 {'product_id': 182, 'product_name': 'Ask Table'},
 {'product_id': 178, 'product_name': 'Audience Snack'},
 {'product_id': 60, 'product_name': 'Bag Table'},
 {'product_id': 161, 'product_name': 'Base Device'},
 {'product_id': 139, 'product_name': 'Base Toy'},
 {'product_id': 158, 'product_name': 'Become Device'},
 {'prod

In [57]:
def get_product_history(cursor, product_id):
    query = "select * from product_inventory_history where product_id = %s order by activity_date desc"
    cursor.execute(query, (product_id,))
    return cursor.fetchall()

In [58]:
get_product_history(cursor, 15)

[{'product_id': 15,
  'product_name': 'Gun Device',
  'supplier_id': 37,
  'activity_date': '2025-04-21',
  'activity_type': 'shipment',
  'quantity': 267,
  'source': 'Supplier Delivery',
  'supplier_name': 'Kaufman Ltd',
  'running_stock': Decimal('425')},
 {'product_id': 15,
  'product_name': 'Gun Device',
  'supplier_id': 37,
  'activity_date': '2025-04-03',
  'activity_type': 'Restock',
  'quantity': 25,
  'source': 'Stock Entry',
  'supplier_name': None,
  'running_stock': Decimal('425')},
 {'product_id': 15,
  'product_name': 'Gun Device',
  'supplier_id': 37,
  'activity_date': '2025-04-01',
  'activity_type': 'Restock',
  'quantity': 84,
  'source': 'Stock Entry',
  'supplier_name': None,
  'running_stock': Decimal('400')},
 {'product_id': 15,
  'product_name': 'Gun Device',
  'supplier_id': 37,
  'activity_date': '2025-03-19',
  'activity_type': 'Restock',
  'quantity': 89,
  'source': 'Stock Entry',
  'supplier_name': None,
  'running_stock': Decimal('316')},
 {'product_id':

In [66]:
def get_pending_reorders(cursor):
     query = """Select r.reorder_id, p.product_name
                from reorders r join products p 
                on p.product_id = r.product_id
             """
     cursor.execute(query)
     return cursor.fetchall()


In [67]:
get_pending_reorders(cursor)

[{'reorder_id': 1, 'product_name': 'Someone Shirt'},
 {'reorder_id': 2, 'product_name': 'Six Table'},
 {'reorder_id': 3, 'product_name': 'Space Toy'},
 {'reorder_id': 4, 'product_name': 'Blue Device'},
 {'reorder_id': 5, 'product_name': 'Mouth Shirt'},
 {'reorder_id': 6, 'product_name': 'School Table'},
 {'reorder_id': 7, 'product_name': 'Four Shirt'},
 {'reorder_id': 8, 'product_name': 'Fast Shirt'},
 {'reorder_id': 9, 'product_name': 'Character Table'},
 {'reorder_id': 10, 'product_name': 'Fact Device'},
 {'reorder_id': 11, 'product_name': 'Return Table'},
 {'reorder_id': 12, 'product_name': 'Scene Table'},
 {'reorder_id': 13, 'product_name': 'Mission Snack'},
 {'reorder_id': 14, 'product_name': 'Old Shirt'},
 {'reorder_id': 15, 'product_name': 'Within Toy'},
 {'reorder_id': 16, 'product_name': 'Lead Toy'},
 {'reorder_id': 17, 'product_name': 'Guy Device'},
 {'reorder_id': 18, 'product_name': 'Bag Table'},
 {'reorder_id': 19, 'product_name': 'However Table'},
 {'reorder_id': 20, 'pro

In [68]:
pending_reorders = get_pending_reorders(cursor)

In [70]:
reorder_ids = [r['reorder_id'] for r in pending_reorders]
reorder_ids

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185

In [71]:
reorder_labels = [f"ID {r['reorder_id']} - {r['product_name']}" for r in pending_reorders]
reorder_labels


['ID 1 - Someone Shirt',
 'ID 2 - Six Table',
 'ID 3 - Space Toy',
 'ID 4 - Blue Device',
 'ID 5 - Mouth Shirt',
 'ID 6 - School Table',
 'ID 7 - Four Shirt',
 'ID 8 - Fast Shirt',
 'ID 9 - Character Table',
 'ID 10 - Fact Device',
 'ID 11 - Return Table',
 'ID 12 - Scene Table',
 'ID 13 - Mission Snack',
 'ID 14 - Old Shirt',
 'ID 15 - Within Toy',
 'ID 16 - Lead Toy',
 'ID 17 - Guy Device',
 'ID 18 - Bag Table',
 'ID 19 - However Table',
 'ID 20 - White Table',
 'ID 21 - Real Snack',
 'ID 22 - Investment Toy',
 'ID 23 - System Snack',
 'ID 24 - Within Toy',
 'ID 25 - Appear Toy',
 'ID 26 - Four Shirt',
 'ID 27 - Thank Shirt',
 'ID 28 - Paper Toy',
 'ID 29 - Floor Toy',
 'ID 30 - Could Device',
 'ID 31 - Bed Toy',
 'ID 32 - Already Snack',
 'ID 33 - Instead Table',
 'ID 34 - Wait Toy',
 'ID 35 - Still Snack',
 'ID 36 - Describe Toy',
 'ID 37 - Guy Device',
 'ID 38 - Full Table',
 'ID 39 - Foreign Table',
 'ID 40 - After Table',
 'ID 41 - During Toy',
 'ID 42 - School Table',
 'ID 43 -

In [72]:
selected_label = 'ID 3 - Space Toy'

In [76]:
print(reorder_labels.index(selected_label))

2


In [74]:
selected_reorder_id = reorder_ids[reorder_labels.index(selected_label)]
selected_reorder_id

3